In [ ]:
!huggingface-cli login

In [2]:
!pip install transformers > /dev/null

In [3]:
!pip install -U datasets > /dev/null
!pip install "datasets[audio]" > /dev/null
!pip install evaluate > /dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-runtime-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-runtime-cu12 12.5.82 w

In [4]:
!pip install jiwer > /dev/null

In [5]:
# Delete local cache of huggingface's datasets
!rm -rf ~/.cache/huggingface/datasets

## Load dataset

### doof-ferb/infore1_25hours - failed

In [ ]:
import os
import zipfile
import requests
from datasets import load_dataset, Audio

In [ ]:
# Step 1: Download the zip file
url = "https://files.huylenguyen.com/datasets/infore/25hours.zip"
zip_path = "25hours.zip"
extract_path = "25hours_data"

if not os.path.exists(zip_path):
    print("Downloading dataset...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
else:
    print("Already downloaded.")

In [ ]:
# Step 2: Unzip the contents (with password)
if not os.path.exists(extract_path):
    print("Extracting dataset with password...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(path=extract_path, pwd=b"BroughtToYouByInfoRe")
else:
    print("Already extracted.")

In [ ]:
# Step 3: Load with Hugging Face `datasets`
# Assuming the dataset includes a JSON or CSV file pointing to audio + transcriptions
from datasets import load_dataset

# Check what's inside
import os
print("Extracted files:")
print(os.listdir(extract_path))

# For example, if it contains a `metadata.json` or `metadata.csv`
# Replace this with the actual file path if different
metadata_file = os.path.join(extract_path, "metadata.json")  # or .csv

# Load the dataset
dataset = load_dataset("json", data_files=metadata_file, split="train")  # or csv

# Decode audio files (assuming there's a column named "audio" pointing to .wav/.flac)
dataset = dataset.cast_column("audio", Audio())
dataset.set_format(type="torch", columns=["audio", "transcription"])  # or your actual text column

# Check one example
print(dataset[0])

### mozilla-foundation/common_voice_17_0

In [ ]:
from datasets import load_dataset

# Load only Vietnamese "train" split
cv_vi = load_dataset("mozilla-foundation/common_voice_17_0", "vi")

README.md:   0%|          | 0.00/12.7k [00:00<?, ?B/s]

common_voice_17_0.py:   0%|          | 0.00/8.19k [00:00<?, ?B/s]

languages.py:   0%|          | 0.00/3.92k [00:00<?, ?B/s]

release_stats.py:   0%|          | 0.00/132k [00:00<?, ?B/s]

The repository for mozilla-foundation/common_voice_17_0 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/mozilla-foundation/common_voice_17_0.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


...


Reading metadata...: 11533it [00:00, 127522.64it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 377it [00:00, 54743.04it/s]


Generating validated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 5135it [00:00, 72488.88it/s]


In [ ]:
print(dataset)

## Training template - original

In [ ]:
from datasets import Dataset
import pandas as pd
from datasets import Audio
import gc

## we will load the both of the data here.
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## we will rename the columns as "audio", "sentence".
## "audio" will hold the file path to the audio file
## "sentence" will hold the text content of the audio file
train_df.columns = ["audio", "sentence"]
test_df.columns = ["audio", "sentence"]

## convert the pandas dataframes to dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

## convert the sample rate of every audio files using cast_column function
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))
test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer
from transformers import WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="Vietnamese", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Vietnamese", task="transcribe")

In [ ]:
def prepare_dataset(examples):
    # compute log-Mel input features from input audio array
    audio = examples["audio"]
    examples["input_features"] = feature_extractor(
        audio["array"], sampling_rate=16000).input_features[0]
    del examples["audio"]
    sentences = examples["sentence"]

    # encode target text to label ids
    examples["labels"] = tokenizer(sentences).input_ids
    del examples["sentence"]
    return examples
train_dataset = train_dataset.map(prepare_dataset, num_proc=1)
test_dataset = test_dataset.map(prepare_dataset, num_proc=1)

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

## lets initiate the data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
import evaluate

metric = evaluate.load("wer")

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# Load a Pre-Trained Checkpoint
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

In [ ]:
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [ ]:
# Define the Training Arguments
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-base-en",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=15000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    # logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

## start the model training
trainer.train()

## Train whisper small tren common-voice-17.0

###### 1. Load dataset tu huggingface

In [8]:
from datasets import load_dataset, Audio

# Load only Vietnamese split
cv_vi = load_dataset("mozilla-foundation/common_voice_17_0", "vi", trust_remote_code=True, streaming=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/12.7k [00:00<?, ?B/s]

common_voice_17_0.py:   0%|          | 0.00/8.19k [00:00<?, ?B/s]

languages.py:   0%|          | 0.00/3.92k [00:00<?, ?B/s]

release_stats.py:   0%|          | 0.00/132k [00:00<?, ?B/s]

...


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 11533it [00:00, 110133.16it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 377it [00:00, 67061.90it/s]


Generating validated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 5135it [00:00, 96229.72it/s]


###### 2. Tạo tập train và tập test

In [9]:
from datasets import concatenate_datasets

# train_dataset = concatenate_datasets([cv_vi["train"], cv_vi["other"]])
# test_dataset = cv_vi["validated"]
train_dataset = cv_vi["train"]
test_dataset = cv_vi["test"]

###### Lọc bỏ các cột không cần thiết, giữ lại các cột quan trọng

In [10]:
keep_columns = ["client_id", "audio", "sentence"]

# For train dataset
train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in keep_columns]
)

# For test dataset
test_dataset = test_dataset.remove_columns(
    [col for col in test_dataset.column_names if col not in keep_columns]
)

###### Cast lại nội dung audio, với sampling rate từ 48 kHz về 16kHz



In [11]:
## convert the sample rate of every audio files using cast_column function
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))
test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

###### Kiểm tra thông số của các tập train và test, gồm tổng số record và tổng thời gian audio

###### a. if dataset is loaded with streaming=True

In [33]:
print(train_dataset)

IterableDataset({
    features: ['client_id', 'audio', 'sentence'],
    num_shards: 1
})


In [36]:
first_sample = next(iter(train_dataset))

Reading metadata...: 2298it [00:00, 16190.74it/s]


In [37]:
audio_array = first_sample["audio"]["array"]
sampling_rate = first_sample["audio"]["sampling_rate"]
total_time = len(audio_array) * 1.0 / sampling_rate
print(total_time)

5.976


In [39]:
def get_record_length_in_seconds(item):
    sample_count = len(item["audio"]["array"])
    sampling_rate = item["audio"]["sampling_rate"]
    item_length_in_seconds = sample_count * 1.0 / sampling_rate
    return item_length_in_seconds

SyntaxError: 'return' outside function (<ipython-input-39-1888707224>, line 6)

In [40]:
count = sum(1 for _ in train_dataset)
print(f"Number of records in train_dataset: {count}")
count = sum(1 for _ in test_dataset)
print(f"Number of records in test_dataset: {count}")

Reading metadata...: 2298it [00:00, 12973.21it/s]


KeyboardInterrupt: 

In [ ]:
total_train_duration_in_seconds = sum(get_record_length_in_seconds(i) for i in train_dataset)
print(f"Total train duration: {total_train_duration_in_seconds}")

###### b. if dataset is loaded with streaming=False

In [12]:
def compute_total_duration(dataset):
    return sum(len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"] for sample in dataset)

total_train_duration = compute_total_duration(train_dataset)
total_test_duration = compute_total_duration(test_dataset)

print(f"Train dataset: {len(train_dataset)} samples, total duration: {total_train_duration / 3600:.2f} hours")
print(f"Test dataset: {len(test_dataset)} samples, total duration: {total_test_duration / 3600:.2f} hours")

Train dataset: 2298 samples, total duration: 2.89 hours
Test dataset: 1274 samples, total duration: 1.35 hours


###### 4. Load các thành phần của mô hình whisper-small

In [13]:
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer
from transformers import WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="Vietnamese", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Vietnamese", task="transcribe")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [14]:
def prepare_dataset(examples):
    # compute log-Mel input features from input audio array
    audio = examples["audio"]
    examples["input_features"] = feature_extractor(
        audio["array"], sampling_rate=16000).input_features[0]
    del examples["audio"]
    sentences = examples["sentence"]

    # encode target text to label ids
    examples["labels"] = tokenizer(sentences).input_ids
    del examples["sentence"]
    return examples
train_dataset = train_dataset.map(prepare_dataset, num_proc=1)
test_dataset = test_dataset.map(prepare_dataset, num_proc=1)

Map:   0%|          | 0/2298 [00:00<?, ? examples/s]

Map:   0%|          | 0/1274 [00:00<?, ? examples/s]

In [15]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

## lets initiate the data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [16]:
import evaluate

metric = evaluate.load("wer")

In [17]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [18]:
# Load a Pre-Trained Checkpoint
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

In [19]:
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [20]:
# Create a callback that will write training logs into a file
import csv
import os
from transformers import TrainerCallback

class CSVLoggerCallback(TrainerCallback):
    def __init__(self, csv_path):
        self.csv_path = csv_path
        os.makedirs(os.path.dirname(csv_path), exist_ok=True)
        with open(self.csv_path, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["step", "epoch", "loss", "learning_rate"])

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            step = state.global_step
            epoch = logs.get("epoch", None)
            loss = logs.get("loss", None)
            learning_rate = logs.get("learning_rate", None)
            with open(self.csv_path, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([step, epoch, loss, learning_rate])

In [21]:
# Define the Training Arguments
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-base-en",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=200,
    max_steps=15000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=200,
    save_total_limit=2,
    eval_steps=200,
    logging_dir="logs",
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[CSVLoggerCallback(csv_path="logs/training_log.csv")]
)

## start the model training
trainer.train()

<ipython-input-21-1795135068>:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...


Step,Training Loss,Validation Loss


Exception ignored in: <function _xla_gc_callback at 0x7c5a73d3a980>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


KeyboardInterrupt: 

In [22]:
model.save_pretrained("trained_model")
tokenizer.save_pretrained("saved_tokenizer")
print("✅ Training Complete! Model saved to 'trained_model'.")

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


✅ Training Complete! Model saved to 'trained_model'.
